<a href="https://colab.research.google.com/github/Bibhuti-MLAI/Time-Series-Forcasting/blob/main/Revenue_Forcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
data_path = r"/content/drive/MyDrive/Alma Better/Revenue Forcasting/bfsi_revenue_forecasting_project/data/BFSI_Revenue_Forecasting_Dataset.xlsx"
RANDOM_STATE = 42

In [2]:
customer = pd.read_excel(data_path, sheet_name="Customer_Master")

In [3]:
customer.head()

,customer_id,customer_name,segment,industry,onboarding_date,credit_limit,risk_grade,geography
0,CUST00056,Client Enterprises 56,Medium,Pharma,15/11/18,10926167.91,C,NaN
1,CUST00052,Client Enterprises 52,Micro,Logistics,07/24/2018,25541292.36,B,Telangana
2,CUST00002,Client Enterprises 2,MSME,Logistics,17-10-2021,29729379.37,A,Maharashtra
3,CUST00089,Client Enterprises 89,SME,IT Services,04/18/2022,14092573.62,A,Delhi NCR
4,CUST00048,Client Enterprises 48,SME,Textiles,03/09/18,29928698.10,A+,Karnataka


In [4]:
invoice = pd.read_excel(data_path, sheet_name="Invoice_Data")
payment = pd.read_excel(data_path, sheet_name="Payment_Data")
repayment =pd.read_excel(data_path, sheet_name="Repayment_Data")

In [5]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113 entries, 0 to 112
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      113 non-null    object 
 1   customer_name    113 non-null    object 
 2   segment          113 non-null    object 
 3   industry         113 non-null    object 
 4   onboarding_date  113 non-null    object 
 5   credit_limit     104 non-null    float64
 6   risk_grade       98 non-null     object 
 7   geography        107 non-null    object 
dtypes: float64(1), object(7)
memory usage: 7.2+ KB


In [6]:
customer.describe().T

,count,mean,std,min,25%,50%,75%,max
credit_limit,104.0,2.464384e+07,1.384461e+07,583718.27,12734032.6,2.558577e+07,35548554.17,49400676.43


In [7]:
customer['customer_id'].count()

np.int64(113)

Clean Customer_master

In [8]:
customer['customer_id']= customer['customer_id'].astype(str).str.strip().str.upper()

In [9]:
customer

,customer_id,customer_name,segment,industry,onboarding_date,credit_limit,risk_grade,geography
0,CUST00056,Client Enterprises 56,Medium,Pharma,15/11/18,10926167.91,C,NaN
1,CUST00052,Client Enterprises 52,Micro,Logistics,07/24/2018,25541292.36,B,Telangana
2,CUST00002,Client Enterprises 2,MSME,Logistics,17-10-2021,29729379.37,A,Maharashtra
3,CUST00089,Client Enterprises 89,SME,IT Services,04/18/2022,14092573.62,A,Delhi NCR
4,CUST00048,Client Enterprises 48,SME,Textiles,03/09/18,29928698.10,A+,Karnataka
...,...,...,...,...,...,...,...,...
108,CUST00029,Client Enterprises 29,MSME,IT Services,20/09/18,8673784.47,A+,TN
109,CUST00096,Client Enterprises 96,Large,IT Services,13-07-2019,19748431.56,NaN,West Bengal
110,CUST00016,Client Enterprises 16,MSME,IT Services,12-Mar-2019,44291816.12,NaN,MH
111,CUST00091,Client Enterprises 91,Small,Auto Ancillary,11/05/2018,35467747.51,A,West Bengal


In [10]:
digits = customer['customer_id'].str.extract(r"CUST0*(\d+)")[0]
customer["customer_id"] = "CUST" + digits.str.zfill(5)
customer["customer_id"].head()

,customer_id
0,CUST00056
1,CUST00052
2,CUST00002
3,CUST00089
4,CUST00048


In [11]:
digits

,0
0,56
1,52
2,2
3,89
4,48
...,...
108,29
109,96
110,16
111,91


In [12]:
customer["customer_name"] = customer["customer_name"].astype(str).str.strip()

In [13]:
customer["onboarding_date"] =pd.to_datetime(customer["onboarding_date"],dayfirst=True,errors="coerce")
customer[["customer_id","customer_name","onboarding_date"]].head()

/tmp/ipykernel_599/2456310456.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  customer["onboarding_date"] =pd.to_datetime(customer["onboarding_date"],dayfirst=True,errors="coerce")


,customer_id,customer_name,onboarding_date
0,CUST00056,Client Enterprises 56,2018-11-15
1,CUST00052,Client Enterprises 52,2018-07-24
2,CUST00002,Client Enterprises 2,2021-10-17
3,CUST00089,Client Enterprises 89,2022-04-18
4,CUST00048,Client Enterprises 48,2018-09-03


In [14]:
from numpy._core.defchararray import title
#standardise free text categories
customer["industry"] = customer["industry"].astype(str).str.strip().str.title()
customer["geography"] = customer["geography"].astype(str).str.strip().str.title()
customer["geography"] = customer["geography"].replace({"Mh":"Maharastra","Tn":"Tamil nadu"})
customer.loc[customer["geography"].isin(["Nan","None"]),"geography"] = np.nan

customer["segment"] = customer["segment"].astype(str).str.strip().str.title()
customer["segment"] = customer["segment"].replace({"Sme":"SME","Msme":"MSME"})
customer[["segment","industry","geography"]].head()

,segment,industry,geography
0,Medium,Pharma,NaN
1,Micro,Logistics,Telangana
2,MSME,Logistics,Maharashtra
3,SME,It Services,Delhi Ncr
4,SME,Textiles,Karnataka


In [15]:
before  = len(customer)
customer = customer.sort_values("customer_id").drop_duplicates(subset="customer_id",keep = "first")
print(f"Customer_master:{before} -> {len(customer)} rows after removing duplicates")

Customer_master:113 -> 110 rows after removing duplicates


In [16]:
#Missing Values
print("Missing Values : ")
print(customer[["geography","credit_limit","risk_grade"]].isna().sum())

Missing Values : 
geography        6
credit_limit     9
risk_grade      15
dtype: int64


In [17]:
customer["geography"] = customer['geography'].fillna("Unknown")
customer["credit_limit"] = customer.groupby("segment")["credit_limit"].transform(lambda x: x.fillna(x.median()))
customer["credit_limit"] = customer["credit_limit"].fillna(customer["credit_limit"].median())
customer['risk_grade'] = customer["risk_grade"].fillna("Unrated")

print("Missing after")
print(customer[["geography","credit_limit","risk_grade"]].isna().sum())

Missing after
geography       0
credit_limit    0
risk_grade      0
dtype: int64


Cleaning Invoice data

In [21]:
invoice["customer_id"] = invoice["customer_id"].astype(str).str.strip().str.upper()
digits = invoice["customer_id"].str.extract(r"CUST0*(\D+)")[0]

invoice["invoice_date"] = pd.to_datetime(invoice["invoice_date"], dayfirst=True, errors = "coerce")
invoice["due_date"] = pd.to_datetime(invoice["due_date"],dayfirst=True,errors = "coerce")
invoice["invoice_status"] = invoice["invoice_status"].astype(str).str.strip().str.title()
invoice["discounting_flag"] = invoice["discounting_flag"].astype(str).str.strip().str.upper()
invoice.head()

/tmp/ipykernel_599/1418042568.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  invoice["invoice_date"] = pd.to_datetime(invoice["invoice_date"], dayfirst=True, errors = "coerce")


,invoice_id,customer_id,invoice_date,invoice_amount,due_date,invoice_status,discounting_flag
0,INV0001348,CUST00072,2023-09-25,60110.08,2023-10-25,Open,Y
1,INV0001541,CUST00083,2022-09-13,60974.11,NaT,Paid,Y
2,INV0001936,CUST00105,2022-03-02,74165.65,2022-04-16,Partially Paid,N
3,INV0000293,CUST00015,2022-08-20,97633.69,NaT,Paid,N
4,INV0000372,CUST00020,2023-05-15,104900.40,NaT,Overdue,Y


In [23]:
before = len(invoice)
invoice = invoice.drop_duplicates()
print(f"invoice_data:removed {before - len(invoice)} extract duplicate rows")

invoice_data:removed 0 extract duplicate rows
